# Demo: Multi Flow SDP with Liquid Clustering and Data Quality
*Local JupyterLab replica of the Databricks Academy demo (Advanced Techniques with Apache Spark Declarative Pipelines).*

In this demo you will:
1. Ingest **two independent sources** into **one** Bronze streaming table using *append flows*
2. Enforce **data quality expectations** (`expect`, `expect_or_drop`, `expect_or_fail`) in Silver
3. Store Silver with **Liquid Clustering** (`CLUSTER BY status, ingest_date`)
4. Build a **Gold** materialized view of business metrics
5. Observe **incremental processing** as new files land, and a **pipeline failure** caused by `expect_or_fail`

> In Databricks you'd attach to a cluster and run the classroom setup. Here the setup cell creates a local Delta-enabled `spark` session.

## A. Classroom Setup
Run this first. It creates `spark`, `display()`, the `%%sql` magic and the `DA` helper object.

In [ ]:
%run ./Includes/Classroom-Setup.py

In [ ]:
DA.reset()   # fresh, dirty starting dataset: 3 files x 50 rows per flow

## B. Explore the raw source files
Each flow lands newline-delimited JSON. Notice the dirty values: null `event_ts`, invalid `status`, negative `amount`.

In [ ]:
raw_a = spark.read.json(DA.paths.source_flow_a)
display(raw_a.limit(10))

In [ ]:
from pyspark.sql import functions as F
display(raw_a.unionByName(spark.read.json(DA.paths.source_flow_b)).select(
    F.count("*").alias("rows"),
    F.sum(F.col("event_ts").isNull().cast("int")).alias("null_event_ts"),
    F.sum((~F.col("status").isin("ACTIVE","INACTIVE","PENDING","ERROR")).cast("int")).alias("bad_status"),
    F.sum((F.col("amount") < 0).cast("int")).alias("negative_amount")))

## C. The pipeline definition
In Databricks the pipeline lives in its own source file and is run by the pipeline engine. Here it's [`pipeline.py`](../pipeline.py) at the project root. Open it next to this notebook and compare:

| Databricks SDP | Local |
|---|---|
| `@dp.table` + 2 x `@dp.append_flow` | `run_bronze()`: one stream per flow, own checkpoint each |
| `@dp.expect*` decorators | `DataQualityManager` rules |
| `cluster_by=["status","ingest_date"]` | `DeltaTable.createIfNotExists(...).clusterBy(...)` |
| `@dp.materialized_view` | `run_gold()` |

In [ ]:
from pipeline import MultiFlowPipeline
pipe = MultiFlowPipeline(spark, DA.paths.working_dir)

for e in pipe.dq.expectations:
    print(f"{e.action:>5}  {e.name:<22} {e.condition}")

## D. Bronze — multi-flow ingestion
Two append flows write into the **same** target table. Each flow keeps its own checkpoint, so flows can be added, removed or backfilled independently.

In [ ]:
pipe.run_bronze()
DA.register_tables(pipe)

In [ ]:
%%sql
SELECT source_flow, count(*) AS rows, count(DISTINCT source_file) AS files
FROM bronze_events
GROUP BY source_flow ORDER BY source_flow

In [ ]:
import os
print(os.listdir(DA.paths.checkpoints))   # one checkpoint per append flow

## E. Silver — data quality expectations + Liquid Clustering

In [ ]:
pipe.run_silver()
DA.register_tables(pipe)

### E1. Expectation metrics (the local "event log")
`warn` rules count violations but keep rows; `drop` rules move rows to quarantine; `fail` rules stop the update.

In [ ]:
display(DA.event_log(pipe)
        .groupBy("expectation", "action")
        .agg(F.sum("total").alias("records"), F.sum("failed").alias("failed_records"))
        .orderBy("expectation"))

In [ ]:
%%sql
SELECT 'bronze' AS layer, count(*) AS rows FROM bronze_events
UNION ALL SELECT 'silver', count(*) FROM silver_events
UNION ALL SELECT 'quarantine', count(*) FROM silver_quarantine

In [ ]:
%%sql
SELECT event_id, status, event_ts, amount, failed_expectations
FROM silver_quarantine LIMIT 10

In [ ]:
%%sql
-- warn-only rule: negative amounts are flagged but still in Silver
SELECT count(*) AS negative_amount_rows_in_silver FROM silver_events WHERE amount < 0

### E2. Verify Liquid Clustering
Check `clusteringColumns` and the `clustering` table feature, then look at the history: `CREATE TABLE` (with clusterBy), then `WRITE`, then `OPTIMIZE` (which does the clustering).

In [ ]:
%%sql
DESCRIBE DETAIL silver_events

In [ ]:
%%sql
DESCRIBE HISTORY silver_events

## F. Gold — materialized view

In [ ]:
pipe.run_gold()
DA.register_tables(pipe)

In [ ]:
%%sql
SELECT * FROM gold_daily_metrics ORDER BY ingest_date, status

## G. Incremental processing
Land a new file in **flow B only** and run the pipeline again. Bronze and Silver process **only the new file** (checkpoints), and Gold is recomputed.

In [ ]:
DA.land_new_files("b")
pipe.run()
display(DA.event_log(pipe).groupBy("batch_id").agg(F.max("total").alias("rows_in_batch")).orderBy("batch_id"))

In [ ]:
%%sql
SELECT source_flow, count(*) AS rows FROM bronze_events GROUP BY source_flow ORDER BY 1

## H. Changing cluster keys (no data rewrite)
Unlike partitioning, Liquid Clustering keys can be changed in place. New writes and the next `OPTIMIZE` use the new keys.

In [ ]:
%%sql
ALTER TABLE silver_events CLUSTER BY (device_id)

In [ ]:
%%sql
OPTIMIZE silver_events

In [ ]:
%%sql
DESCRIBE DETAIL silver_events

## I. `expect_or_fail` stops the update
Land a file containing a row with `event_id = NULL`. The Silver update fails and **nothing from that micro-batch is committed**.

In [ ]:
from pipeline import ExpectationFailed
DA.land_new_files("a", null_event_id=True)
pipe.run_bronze()
try:
    pipe.run_silver()
except Exception as exc:
    msg = next((l for l in str(exc).splitlines() if "expect_or_fail" in l), str(exc))
    print("PIPELINE UPDATE FAILED ->", msg.strip())

In [ ]:
%%sql
-- the bad row reached Bronze, but Silver did not change
SELECT (SELECT count(*) FROM bronze_events WHERE event_id IS NULL) AS null_ids_in_bronze,
       (SELECT count(*) FROM silver_events)                        AS silver_rows

## J. Clean up
Run this to reset the lesson data (like `DA.cleanup()` in the course).

In [ ]:
DA.reset()
spark.catalog.clearCache()